# Exploratory Data Analysis (EDA) - Stock Price Prediction & Technical Analysis

This notebook provides a comprehensive exploratory analysis of historical stock price data (using **RELIANCE.NS** as the primary example) and visualizes the engineered technical features. It covers:
1. Data loading and basic statistical summaries
2. Price and volume trend analysis
3. Daily returns distributions
4. Trend indicators (Moving Averages, MACD, ADX)
5. Momentum indicators (RSI)
6. Volatility indicators (Bollinger Bands, ATR)
7. Volume indicators (OBV)
8. Correlation analysis of technical indicators
9. Seasonality analysis (Day of week returns)
10. Autocorrelation & stationarity checks (ACF/PACF)

In [ ]:
import os
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

### 1. Download Stock Data
Let's fetch historical data for **RELIANCE.NS** from yfinance (from 2018-01-01 to present) and show the first few rows.

In [ ]:
ticker = "RELIANCE.NS"
df_raw = yf.download(ticker, start="2018-01-01", progress=False)

# Flatten multi-index if necessary (newer yfinance versions)
if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = [col[0] for col in df_raw.columns]

df_raw = df_raw.reset_index()
df_raw.columns = [col.lower() for col in df_raw.columns]
print(f"Loaded {len(df_raw)} records.")
df_raw.head()

### 2. Basic Statistical Summary

In [ ]:
df_raw.describe()

### 3. Feature Engineering Fallback
Let's calculate the technical indicators manually so the notebook runs independently of external technical analysis libraries.

In [ ]:
df = df_raw.copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Daily Returns
df['daily_return'] = df['close'].pct_change() * 100

# Moving Averages
df['sma_7'] = df['close'].rolling(window=7).mean()
df['sma_21'] = df['close'].rolling(window=21).mean()
df['sma_50'] = df['close'].rolling(window=50).mean()
df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()
df['ema_26'] = df['close'].ewm(span=26, adjust=False).mean()

# RSI (14)
delta = df['close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()
for i in range(14, len(df)):
    avg_gain.iloc[i] = (avg_gain.iloc[i-1] * 13 + gain.iloc[i]) / 14
    avg_loss.iloc[i] = (avg_loss.iloc[i-1] * 13 + loss.iloc[i]) / 14
rs = avg_gain / avg_loss.replace(0, 1e-9)
df['rsi_14'] = 100 - (100 / (1 + rs))

# MACD
df['macd'] = df['ema_12'] - df['ema_26']
df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
df['macd_hist'] = df['macd'] - df['macd_signal']

# Bollinger Bands (20, 2)
df['bb_middle'] = df['close'].rolling(window=20).mean()
bb_std = df['close'].rolling(window=20).std()
df['bb_upper'] = df['bb_middle'] + (bb_std * 2)
df['bb_lower'] = df['bb_middle'] - (bb_std * 2)

# ATR (14)
high_low = df['high'] - df['low']
high_close = (df['high'] - df['close'].shift()).abs()
low_close = (df['low'] - df['close'].shift()).abs()
tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df['atr_14'] = tr.rolling(window=14).mean()
for i in range(14, len(df)):
    df.loc[i, 'atr_14'] = (df.loc[i-1, 'atr_14'] * 13 + tr.iloc[i]) / 14

# OBV
df['obv'] = (np.sign(df['close'].diff()).fillna(0) * df['volume']).cumsum()

# Day of week
df['day_of_week'] = df['date'].dt.day_name()
df['day_num'] = df['date'].dt.dayofweek

# Drop NaNs for indicators
df_clean = df.dropna().reset_index(drop=True)
print(f"Cleaned dataset size: {len(df_clean)} rows.")

### 4. Visualizations

#### Visualization 1: Historical Closing Price Trend

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df['date'], df['close'], color='#1b3a4b', label='Closing Price', linewidth=1.5)
plt.title(f"{ticker} Historical Closing Price (2018-Present)")
plt.xlabel("Date")
plt.ylabel("Price (INR)")
plt.legend()
plt.show()

#### Visualization 2: Historical Volume Trend

In [ ]:
plt.figure(figsize=(14, 4))
plt.bar(df['date'], df['volume'], color='#3a86c8', alpha=0.6, width=1.0)
plt.title(f"{ticker} Historical Volume")
plt.xlabel("Date")
plt.ylabel("Volume")
plt.show()

#### Visualization 3: Daily Returns Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df_clean['daily_return'], kde=True, bins=100, color='#2a9d8f')
plt.title(f"{ticker} Daily Returns Distribution (%)")
plt.xlabel("Daily Return (%)")
plt.ylabel("Frequency")
plt.xlim(-8, 8)  # Clip outliers for visualization
plt.show()

#### Visualization 4: Rolling Moving Averages (7, 21, 50 SMA)

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(df['date'].tail(300), df['close'].tail(300), color='#2b2d42', label='Close', alpha=0.6, linewidth=1.5)
plt.plot(df['date'].tail(300), df['sma_7'].tail(300), color='#ef233c', label='SMA 7', linewidth=1)
plt.plot(df['date'].tail(300), df['sma_21'].tail(300), color='#ffb703', label='SMA 21', linewidth=1.2)
plt.plot(df['date'].tail(300), df['sma_50'].tail(300), color='#2a9d8f', label='SMA 50', linewidth=1.5)
plt.title(f"{ticker} Rolling Moving Averages (Last 300 Days)")
plt.xlabel("Date")
plt.ylabel("Price (INR)")
plt.legend()
plt.show()

#### Visualization 5: Relative Strength Index (RSI 14)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(df['date'].tail(200), df['close'].tail(200), color='#2b2d42', label='Close')
ax1.set_title(f"{ticker} Closing Price & RSI Indicator")
ax1.set_ylabel("Price (INR)")
ax1.legend()

ax2.plot(df['date'].tail(200), df['rsi_14'].tail(200), color='#8338ec', label='RSI 14')
ax2.axhline(70, color='#e63946', linestyle='--', alpha=0.7, label='Overbought (70)')
ax2.axhline(30, color='#2a9d8f', linestyle='--', alpha=0.7, label='Oversold (30)')
ax2.set_ylabel("RSI Value")
ax2.set_xlabel("Date")
ax2.set_ylim(10, 90)
ax2.legend(loc='lower left')
plt.show()

#### Visualization 6: MACD (Moving Average Convergence Divergence)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(df['date'].tail(200), df['close'].tail(200), color='#2b2d42', label='Close')
ax1.set_title(f"{ticker} Closing Price & MACD")
ax1.set_ylabel("Price (INR)")

ax2.plot(df['date'].tail(200), df['macd'].tail(200), color='#1d3557', label='MACD Line', linewidth=1.2)
ax2.plot(df['date'].tail(200), df['macd_signal'].tail(200), color='#e63946', label='Signal Line', linewidth=1.2)
colors = ['#2a9d8f' if x > 0 else '#e63946' for x in df['macd_hist'].tail(200)]
ax2.bar(df['date'].tail(200), df['macd_hist'].tail(200), color=colors, label='Histogram', alpha=0.5, width=1.0)
ax2.set_ylabel("MACD Value")
ax2.set_xlabel("Date")
ax2.legend(loc='lower left')
plt.show()

#### Visualization 7: Bollinger Bands

In [ ]:
plt.figure(figsize=(14, 7))
tail_df = df.tail(200)
plt.plot(tail_df['date'], tail_df['close'], color='#2b2d42', label='Close', linewidth=1.5)
plt.plot(tail_df['date'], tail_df['bb_upper'], color='#457b9d', linestyle='--', label='Upper Band', linewidth=1)
plt.plot(tail_df['date'], tail_df['bb_middle'], color='#e63946', label='Middle Band (20 SMA)', linewidth=1)
plt.plot(tail_df['date'], tail_df['bb_lower'], color='#457b9d', linestyle='--', label='Lower Band', linewidth=1)
plt.fill_between(tail_df['date'], tail_df['bb_lower'], tail_df['bb_upper'], color='#a8dadc', alpha=0.15)
plt.title(f"{ticker} Bollinger Bands (20, 2)")
plt.xlabel("Date")
plt.ylabel("Price (INR)")
plt.legend()
plt.show()

#### Visualization 8: Correlation Matrix of Indicators

In [ ]:
cols_to_correlate = [
    'close', 'volume', 'daily_return', 'sma_7', 'sma_21', 'sma_50', 
    'rsi_14', 'macd', 'macd_hist', 'bb_upper', 'bb_lower', 'atr_14', 'obv'
]
corr = df_clean[cols_to_correlate].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, square=True)
plt.title(f"{ticker} Indicator Correlation Heatmap", fontsize=16)
plt.show()

#### Visualization 9: Seasonality - Daily Returns by Day of Week

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
plt.figure(figsize=(10, 6))
sns.boxplot(x='day_of_week', y='daily_return', data=df_clean, order=order, palette="Set2")
plt.title(f"{ticker} Daily Returns by Day of the Week")
plt.xlabel("Day of the Week")
plt.ylabel("Daily Return (%)")
plt.ylim(-5, 5)  # Zoom in on main distribution
plt.show()

#### Visualization 10: Time Series Autocorrelation (ACF & PACF)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# ACF on Close Price
plot_acf(df_clean['close'], lags=40, ax=ax1, title="Autocorrelation of Close Price")
ax1.set_xlabel("Lags")

# PACF on Close Price
plot_pacf(df_clean['close'], lags=40, ax=ax2, title="Partial Autocorrelation of Close Price", method='yule_walker')
ax2.set_xlabel("Lags")

plt.show()